# Neighborhood analysis {#sec-img-neighborhood-analysis}



## Preamble

### Introduction

Neighborhood analysis aims to investigate the composition of and interaction between cells and cell types within small regions of tissue, which are referred to as neighborhoods or niches.


### Dependencies

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(patchwork)
library(RColorBrewer)
library(BiocParallel)
library(SpatialExperiment)

In [ ]:
# load data from previous section
spe <- readRDS("img-spe_cl.rds")
spe$k <- colLabels(spe)

In [ ]:
# basic theme for spatial plots
theme_xy <- list(
    coord_equal(expand=FALSE), 
    theme_void(), theme(
        plot.margin=margin(l=5),
        legend.key=element_blank(),
        panel.background=element_rect(fill="black")))

## Nearest neighbors

Custom spatial analyses may rely on identifying nearest neighbors (NNs) of cells.
We recommend `r BiocStyle::CRANpkg("RANN")` for this purpose, which finds NNs in **$O(N\log N)$ time for $N$ cells** (c.f., conventional approaches would take $O(N^2)$ time) by relying on a Approximate Near Neighbor (ANN) C++ library.
Furthermore, there is support for **exact, approximate, and fixed-radius searchers**.
The latter is of particular interest in biology; e.g., one might require $k$NNs to lie within a biologically sensible distance as to avoid consideration of cells that are far-off, especially in sparse regions or at tissue borders.

As a toy example, we here compute the $k$NNs between a pair of subpopulations, with and without thresholding on NN distances (`searchtype="radius"`).

- For the first approach, each cell will receive $k$ neighbors exactly,  
but these may lie within an arbitrary distance.

- For the second approach, cells will receive $\leq k$ neighbors,  
depending on how many cells lie within a radius $r$.

In [ ]:
library(RANN)
k <- 10 # num. neighbors
r <- 50 # dist. threshold
i <- spe$k == 1 # source
j <- spe$k == 4 # target
xy <- spatialCoords(spe)
# k-NN search: all cells have k neighbors
ns_k <- nn2(xy[j, ], xy[i, ], k=k)
is_k <- ns_k$nn.idx
all(rowSums(is_k > 0) == k) 
# w/ fixed-radius: cells have 0-k neighbors
ns_r <- nn2(xy[j, ], xy[i, ], k=k, searchtype="radius", r=r)
is_r <- ns_r$nn.idx
range(rowSums(is_r > 0))

The neighbors obtained via fixed-radius search (right) are less scattered than those obtained for unlimited distances (left); the former are arguably more meaningful in a biological context:

In [ ]:
#| code-fold: true
df <- data.frame(xy, colData(spe))
p0 <- ggplot(df, aes(x_centroid, y_centroid)) + 
    geom_point(data=df, col="navy", shape=16, size=0) +
    geom_point(data=df[i, ], col="magenta", shape=16, size=0) 
p1 <- p0 + geom_point(
    data=df[which(j)[is_k], ], 
    col="gold", shape=16, size=0) +
    ggtitle("k-nearest neighbors")
p2 <- p0 + geom_point(
    data=df[which(j)[is_r], ], 
    col="gold", shape=16, size=0) +
    ggtitle("fixed-radius search")
(p1 | p2) + plot_layout(nrow=1) & theme_xy

::: {.callout-note collapse="true" title="exhaustive fixed-radius search"}
Note that, we could also set a very large `k` in order to identify *all* neighbors within a radius `r`.
In order to prevent unnecessarily costly searches, it is sensible to estimate how many neighbors we would expect, and to set `k` accordingly. 
`nn2()` will otherwise find each cells' $k$NNs, and set the indices of those with a distance $>r$ to 0.

As an exemplary approach, we here sample 1,000 cells to estimate the highest number of NNs obtained, considering *half* of all target cells as potential NNs:

In [ ]:
# test search
.i <- sample(which(i), 1e3)
ns <- nn2(
    xy[j, ], xy[.i, ], 
    k=round(sum(j)/2), 
    searchtype="radius", r=r)
(.k <- max(rowSums(ns$nn.idx > 0)))

For our actual search, we then set `k` to be twice our estimate.
As a final spot-check, we make sure that all cells have fewer than `k` NNs, since we might otherwise be missing some.

In [ ]:
# real search
ns <- nn2(
    xy[j, ], xy[i, ], 
    k=k <- ceiling(2*.k), 
    searchtype="radius", r=r)
max(rowSums(ns$nn.idx > 0)) < k

:::


## Spatial contexts

Spatial niche analysis aims at identifying regions of homogeneous composition by grouping cells based on their microenvironment.
To this end, methods such as `r BiocStyle::Biocpkg("imcRtools")` [@Windhager2023-imcRtools] rely on a $k$-nearest-neighbor ($k$NN) graph (based on Euclidean cell-to-cell distances), and clustering cells using common clustering algorithms (according to their neighborhood's subpopulation frequencies).

Here, we demonstrate how to identify spatial contexts based on $k$-means clustering on cluster frequencies among (Euclidean) $k$NNs.
We recommend readers consult the `imcRtools` documentation for a much wider range of visualizations and downstream analyses in this context.

In [ ]:
library(imcRtools)
# construct kNN-graph based on Euclidean distances
sqe <- buildSpatialGraph(spe, 
    coords=spatialCoordsNames(spe),
    img_id="sample_id", type="knn", k=10)
# compute cluster frequencies among each cell's kNNs
sqe <- aggregateNeighbors(sqe, 
    colPairName="knn_interaction_graph", 
    aggregate_by="metadata", count_by="k")
# view composition of 1st cell's kNNs
unlist(sqe$aggregatedNeighbors[1, ]) 
# cluster cells by neighborhood compositions
ctx <- kmeans(sqe$aggregatedNeighbors, centers=5)
table(sqe$ctx <- factor(ctx$cluster))

Let's quickly view the subpopulation composition of each spatial context:

In [ ]:
df <- data.frame(spatialCoords(sqe), colData(sqe))
round(100*with(df, prop.table(table(k, ctx), 2)), 2)

Secondly, let's visualize the obtained spatial contexts in space:

In [ ]:
#| code-fold: true
pal_k <- unname(pals::trubetskoy(nlevels(df$k)))
pal_c <- c("blue", "cyan", "gold", "magenta", "maroon")
ggplot(df, aes(x_centroid, y_centroid, col=k)) + 
    scale_color_manual(values=pal_k) +
ggplot(df, aes(x_centroid, y_centroid, col=ctx)) + 
    scale_color_manual(values=pal_c) +
plot_layout(nrow=1) &
    geom_point(shape=16, size=0) &
    guides(col=guide_legend(override.aes=list(size=2))) &
    theme_xy & theme(legend.key.size=unit(0.5, "lines"))

## Co-localization

`r BiocStyle::Biocpkg("hoodscanR")` [@Liu2025-hoodscanR] also relies on a (Euclidean) $k$NN graph to estimate the probability of each cell associating with its NNs.
The resulting probability matrix (rows=cells, columns=NNs) can, in turn, be used to assess co-occurrence of subpopulations. 

In [ ]:
library(hoodscanR)
sqe <- readHoodData(spe, anno_col="k")
nbs <- findNearCells(sqe, k=100)
mtx <- scanHoods(nbs$distance)      
grp <- mergeByGroup(mtx, nbs$cells) 
sqe <- mergeHoodSpe(sqe, grp)       

To perform neighborhood co-localization analysis, `plotColocal()` computes the Pearson correlation of probability distribution between cells.
Here, **high/low values indicate attraction/repulsion** between clusters:

In [ ]:
library(pheatmap)
cor <- plotColocal(sqe, pm_cols=colnames(grp), return_matrix=TRUE)
pal <- colorRampPalette(rev(hcl.colors(9, "Roma")))(100)
pheatmap(cor, 
    cellwidth=15, cellheight=15, 
    treeheight_row=5, treeheight_col=5,
    col=pal, breaks=seq(-1, 1, length=100))

::: {.callout-note title="measuring local mixing" collapse="true"}

Downstream, `calcMetrics()` can be used to calculate cell-level [entropy](https://en.wikipedia.org/wiki/entropy_(information_theory)) and [perplexity](https://en.wikipedia.org/wiki/perplexity),
which both measure the mixing of cellular neighborhoods.
Here, **low/high values indicate heterogeneity/homogeneity** of a cell's local neighborhood:

In [ ]:
sqe <- calcMetrics(sqe, pm_cols=colnames(grp))

In [ ]:
#| code-fold: true
df <- data.frame(colData(sqe), k=spe$k, spatialCoords(spe))
vs <- c("perplexity", "entropy")
fd <- df |>
    pivot_longer(all_of(vs)) |>
    group_by(name) |> mutate_at("value", scale) 
# threshold at 2 SDs for clearer visualization
fd$value[i] <- 2*sign(fd$value[i <- abs(fd$value) > 2])
ggplot(fd, aes(x_centroid, y_centroid, col=value)) +
    facet_grid(~name) + geom_point(shape=16, size=0) + 
    theme_xy + theme(legend.key.size=unit(0.5, "lines")) +
    scale_color_gradient2("z-scaled\nvalue", low="cyan", mid="navy", high="magenta") 

Stratifying these values by subpopulation, we can observe that clusters forming distinct aggregates in space are lowest in entropy/perplexity (i.e., the most homogeneous locally):

In [ ]:
#| code-fold: true
ggplot(fd, aes(k, value, fill=k)) +
    facet_wrap(~name) + 
    scale_fill_manual(values=pal_k) + 
    geom_boxplot(outlier.stroke=0, key_glyph="point") +
    scale_y_continuous("z-scaled value", limits=c(-2, 2)) +
    guides(fill=guide_legend(override.aes=list(shape=21, size=2))) +
    theme_bw() + theme(
        axis.title.x=element_blank(),
        panel.grid.minor=element_blank(),
        legend.key.size=unit(0.5, "lines"))

:::

## Spatial density-based analysis

`r BiocStyle::Biocpkg("scider")` [@Li2025-scider] defines spatial domains as regions of interest (ROIs) while preserving the tissue structure through spatial density analysis. 
Kernel density estimation (KDE) is used to describe the spatial distributions of cells, and ROIs are identified based on the spatial density of some cell types of interest (COIs).
One can also calculate the spatial density of transcript expression of a gene or a gene set, so that gene (set)-specific ROIs can be identified. 
ROIs can then be used for downstream analysis such as cell type co-localization, cell type composition and differential expression (DE) analysis. 
All functions operate on `SpatialExperiment` objects, allowing seemless integration. 
Using the Xenium breast carcinoma dataset, we demonstrate a standard `r BiocStyle::Biocpkg("scider")` workflow. 

In [ ]:
#| message: false
# dependencies
library(scider)
library(OSTA.data)
library(SpatialExperimentIO)
# retrieve data from OSF repo &
# read into 'SpatialExperiment'
id <- "Xenium_HumanBreast1_Janesick"
pa <- OSTA.data_load(id, mol=FALSE)
dir.create(td <- tempfile())
unzip(pa, exdir=td)
spe <- readXeniumSXE(td, addTx=FALSE)

In this example, we define ROIs based on the celluar density of tumor cells, that is, DCIS_1, DCIS_2 and invasive tumor cells are considered as the COI. 
Firstly, we need to add cell type annotations to `colData`, where cell type annotations were obtained from the original publication [@Janesick2023-high-res]. 

In [ ]:
#| fig-width: 5
#| fig-height: 3.5
#| fig-cap: "Spatial plot of the Xenium breast carcinoma sample, replicate 1."

anno <- read.csv(file.path(td, "annotation.csv"))
spe$cell_type <- anno$Annotation[match(spe$cell_id, anno$Barcode)]

scider::plotSpatial(spe, group="cell_type", pt.alpha=1)

We first estimate the spatial density of each cell type using the `gridDensity()` function. 

In [ ]:
spe <- gridDensity(spe, grid.length.x=50, bandwidth=25)
metadata(spe)$grid_density[1:4, 1:8]

The density value represents the expected number of COI cells on each grid, and can be visualized by the `plotDensity()` function. 
For example, here we visualize the density of COI cells:

In [ ]:
#| fig-width: 5
#| fig-height: 3.5
#| fig-cap: "Heatmap of the spatial density of COI cells."

coi <- c("DCIS_1", "DCIS_2", "Invasive_Tumor")
scider::plotDensity(spe, coi=coi, probs=0.5) +
  ggtitle("Spatial density of tumour cells")

ROI detection is performed using a graph-based approach as described in @Li2025-scider. 
Only grids with $\geq 1$ expected COI cell (estimated COI density $\geq 1$) are retained in ROI detection. 

In [ ]:
#| fig-width: 5
#| fig-height: 3.5
#| fig-cap: "ROIs detected based on COI density."
#| message: false

spe <- findROI(spe, coi=coi, min.density=1)
scider::plotROI(spe, roi=coi) + 
  ggtitle("Tumour cell-based ROIs")

These ROIs can then be used for downstream analysis, such as cell type colocalization, spatial domain clustering, and differential expression (DE). 
Here as an example, we demonstrate an ROI-level cell type co-localization analysis. 

### Cell type co-localization on the ROI level

`r BiocStyle::Biocpkg("scider")` performs cell type co-localization analysis by calculating the correlation between the spatial densities of every two cell types. 

In [ ]:
#| fig-width: 3.5
#| fig-height: 3
#| fig-cap: "Cell type colocalization in tumor-specific ROIs."
#| message: false

spe_cor <- corDensity(spe)
scider::plotCorHeatmap(spe_cor)

We see that immune and stromal cells are positively colocalized within their respective groups, whereas myoepithelial, DCIS, and invasive tumor populations form distinct, negatively correlated clusters.

### Cell type composition analysis based on density contours

Additionally, contour levels can be calculated from the estimated COI density, which is useful for cell type composition analysis. 

In [ ]:
#| fig-width: 5
#| fig-height: 3.5
#| fig-cap: "Spatial plot of contour lines calculated from the spatial density of tumor cells."
#| message: false

spe <- getContour(spe, coi=coi, bins=10)
scider::plotContour(spe, coi=coi, line.width=0.5)

We can then assign each cell to its corresponding contour level according to their spatial coordinates, and visualize the cell type composition at each COI contour level using the `plotCellCompo()` function. 

In [ ]:
#| fig-width: 4
#| fig-height: 2.5
#| fig-cap: "Cell type composition at each contour level of COI cells across all ROIs."
#| message: false

spe <- allocateCells(spe, contour=coi, to.roi=TRUE)
scider::plotCellCompo(spe, contour=coi, self.included=FALSE)

For example, we see a decreasing proportion of stromal cells, and an increasing proportion of unlabeled cells as tumor cell density increases. 
Normally, we would have filtered out unlabeled cells in preprocessing. 
It is also useful to investigate how cell type composition changes to COI densities within each ROI:

In [ ]:
#| fig-width: 10
#| fig-height: 8
#| fig-cap: "Cell type composition at each contour level of COI cells within each ROI."
#| message: false

scider::plotCellCompo(spe, contour=coi, self.included=FALSE, roi=coi)

ROIs can also be used to perform other types of downstream analysis, such as gene expression clustering, DE analysis and pathway analysis by pseudo-bulking cells located within each ROI, using the `spe2PB()` function. 
Given the pseudo-bulk samples, `r BiocStyle::Biocpkg("scider")` is fully compatible with all `r BiocStyle::Biocpkg("limma")` and `r BiocStyle::Biocpkg("edgeR")` functionalities, allowing arbitrarily complex experimental design to answer various biological questions.
We recommend to perform such DE analyses using the `voomLmFit()` pipeline [@Baldoni2025-diffSplice] or the quasi-likelihood pipeline in `r BiocStyle::Biocpkg("edgeR")` [@Chen2025-edgeR-v4].
Examples of DE and pathway analyses can be found in @Li2025-scider. 
More case studies will also be available at https://github.com/ChenLaboratory/scider. 


## Appendix

### References {.unnumbered}